# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a step-by-step guide for loading and exploring the FAIR² dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets and fields with their `@id` values.

In [ ]:
# List all record sets by `@id` and show their fields' `@id`s
print("Available Record Sets and Fields:")
record_sets = metadata.record_sets
for rs in record_sets:
    print(f"- RecordSet @id: {rs['@id']}  |  Name: {rs.get('name', '[no name]')}")
    if 'fields' in rs:
        print("  Fields:")
        for field in rs['fields']:
            print(f"    - Field @id: {field['@id']}  |  Name: {field.get('name', '[no name]')}")
    print()
# Save record set IDs for later
record_set_ids = [rs['@id'] for rs in record_sets]

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Record sets and field `@id`s are used.

In [ ]:
# Extract data from each record set
dataframes = {}

# For demonstration, print available record set IDs and use the first for EDA
print('Available record set @ids:', record_set_ids)
selected_record_set_id = record_set_ids[0]
print('Using record set:', selected_record_set_id)

# Load records for each record set and create DataFrames
for rs_id in record_set_ids:
    records = list(dataset.records(record_set=rs_id))
    dataframes[rs_id] = pd.DataFrame(records)

print('Columns in selected DataFrame:')
print(dataframes[selected_record_set_id].columns.tolist())
dataframes[selected_record_set_id].head()

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps like filtering, normalization, and grouping for initial exploration. Update field `@id`s as identified above.

In [ ]:
# Set a numeric field and a group field by their column (field @id) names as present in the DataFrame
df = dataframes[selected_record_set_id]
print("Available columns:", df.columns.tolist())

# Guessing field IDs - update as appropriate based on the 'Data Overview' step output.
numeric_field = None
for col in df.columns:
    # Usually, fields with 'coverage', 'count', 'MW', or 'abundance' are numeric.
    if any(word in col.lower() for word in ['coverage', 'count', 'abund', 'mw', 'weight', 'score', 'number']):
        numeric_field = col
        break
if not numeric_field:
    # Fallback, use first numeric-like column
    for col in df.columns:
        if pd.api.types.is_numeric_dtype(df[col]):
            numeric_field = col
            break

print(f"Selected numeric field: {numeric_field}")

threshold = None
if numeric_field is not None:
    # Pick a threshold – e.g., 10th percentile
    try:
        threshold = df[numeric_field].dropna().quantile(0.1)
    except Exception as e:
        threshold = 0

    filtered_df = df[df[numeric_field] > threshold].copy()
    print(f"Filtered records with {numeric_field} > {threshold}:")
    print(filtered_df.head())

    # Normalize the numeric field
    filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
    print(f"Normalized {numeric_field} for filtered records:")
    print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

    # Group by a categorical field if available
    group_field = None
    for col in df.columns:
        # Choose any field likely to be categorical
        if col != numeric_field and (df[col].dtype == object or pd.api.types.is_categorical_dtype(df[col])):
            group_field = col
            break
    if group_field:
        grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().to_frame()
        print(f"Grouped data by {group_field} (mean of {numeric_field}):")
        print(grouped_df.head())
else:
    print("No numeric field found for EDA.")

## 5. Visualization
Visualize the distribution of the selected numeric field and its relationship with a categorical variable, if available.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_field and not filtered_df.empty:
    plt.figure(figsize=(6,3))
    sns.histplot(filtered_df[numeric_field], bins=30, kde=True)
    plt.title(f"Distribution of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.tight_layout()
    plt.show()

    if group_field:
        # Visualize boxplot by group
        plt.figure(figsize=(8,3))
        sns.boxplot(x=filtered_df[group_field], y=filtered_df[numeric_field])
        plt.title(f"{numeric_field} by {group_field}")
        plt.xlabel(group_field)
        plt.ylabel(numeric_field)
        plt.tight_layout()
        plt.show()
else:
    print("Insufficient data for visualization.")

## 6. Conclusion
This notebook demonstrated how to explore protein abundance and related features in human extracellular vesicle samples using `mlcroissant`. 

Key steps:
- Loaded metadata and records with `mlcroissant` using the dataset's Croissant schema URL
- Inspected available record sets, fields, and their `@id`s
- Loaded the main record set into a DataFrame for further analysis
- Performed basic exploratory data analysis: filtering, normalization, and grouping
- Visualized numeric field distributions

Refer to the [FAIR² dataset description](https://sen.science/doi/10.71728/senscience.vq4a-28xa/) for more information and advanced use cases.